---
### **Data preprocessing**

원자료(Raw data)를 정리하고 결측치·이상치를 보정한 후, 수익률·거래규모·팩터 등 포트폴리오 구성에 필요한 데이터를 산출하여 분석 및 전략 수립이 가능한 형태로 변환

---
#### **1. 팩터 데이터 전처리 및 변환**

**`1. 자산가격결정모형에 기반한 수익률 분석`** 및 **`2. 투자 요인 산출을 위한 고유 변동성 계산(해당 예시전략의 경우)`** 을 위해 지수 형태의 raw data를 전처리하여 월간 수익률 형태의 팩터 데이터셋을 준비

---
**데이터 로드**

In [3]:
import pandas as pd

factors = pd.read_csv("input/factors.csv", index_col=0, parse_dates=True)  # 팩터(MKT, SMB, HML, MOM, RF)

factors.tail()

,KOSPI,SMB,HML,MOM,RF
Date,,,,,
2025-09-07,3205.12,113.61,1948.04,224.60,2.57
2025-09-08,3219.59,113.91,1942.88,225.33,2.56
2025-09-09,3260.05,113.29,1961.62,226.26,2.54
2025-09-10,3314.53,112.58,1974.94,229.30,2.55
2025-09-11,3344.20,111.95,1955.01,229.33,2.54


---
**전처리 함수 정의**

In [4]:
import numpy as np

# 전처리 함수 정의
def make_ff_factors(factors, freq="D", annual_rf=True):
    """
    factors: DataFrame with columns ['KOSPI','SMB','HML','MOM','RF']
    freq: 'D' (일간) 또는 'M' (월간)
    """
    
    df = factors.copy()

    # 0. resampling
    if freq == "M":
        df = df.resample('ME').last()
    
    # 1. 지수 → 수익률 변환
    ret_cols = ['KOSPI','SMB','HML','MOM']
    df[ret_cols] = df[ret_cols].pct_change()
    
    # 2. 무위험금리 변환 (연율 → 일/월 수익률)
    df['RF'] = df['RF'] / 100  # % → 소수화 (예: 3.5% → 0.035)
    
    if freq == "D":
        trading_days = 252
        df['RF'] = (1 + df['RF']) ** (1/trading_days) - 1
    elif freq == "M":
        df['RF'] = (1 + df['RF']) ** (1/12) - 1
    
    # 4. 컬럼 정리
    df = df[['KOSPI','SMB','HML','MOM','RF']].dropna()
    
    return df

---
**데이터 계산**

In [5]:
factors_monthly = make_ff_factors(factors, freq="M")[:-1]

factors_monthly.tail()

,KOSPI,SMB,HML,MOM,RF
Date,,,,,
2025-04-30,0.030426,0.054326,-0.011127,0.054588,0.002231
2025-05-31,0.055175,-0.017765,0.061506,0.047865,0.002133
2025-06-30,0.138649,-0.067961,0.028903,0.011075,0.002109
2025-07-31,0.056562,-0.043851,-0.025323,-0.064316,0.002068
2025-08-31,-0.018312,-0.002987,-0.001854,-0.008658,0.002084


---
**팩터 데이터 저장**

In [6]:
# Output 폴더
factors_monthly.to_csv("./output/factors_monthly.csv", index=True, encoding="utf-8-sig")

# 데이터가 필요한 단계의 input 폴더
factors_monthly.to_csv("../Step2 투자 요인 산출 (Factor construction)/input/factors_monthly.csv", index=True, encoding="utf-8-sig")
factors_monthly.to_csv("../Step4 성과 평가 (Performance evalution)/input/factors_monthly.csv", index=True, encoding="utf-8-sig")

---
#### **2. 초과 수익률 계산**

**투자 요인 산출을 위한 고유 변동성(ivol) 계산**을 위해 수정주가 데이터로 초과수익률 데이터 생성

---
**데이터 로드**

In [30]:
import pandas as pd

adj_close        = pd.read_csv("./input/adj close.csv", index_col=0, parse_dates=True)         # 수정주가

adj_close.tail()

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,기아,KB금융,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
2025-08-31,69700.0,269000.0,352000.0,1001000.0,884000.0,220000.0,520000.0,105800.0,108200.0,61700.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-01,67600.0,256000.0,349000.0,996000.0,916000.0,220500.0,510000.0,106500.0,107100.0,59600.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-02,69100.0,260500.0,350000.0,997000.0,933000.0,220000.0,514000.0,107200.0,108800.0,60400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-03,69800.0,262500.0,348500.0,1012000.0,941000.0,221500.0,505000.0,107000.0,110200.0,62400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09-04,70100.0,265500.0,351000.0,1017000.0,931000.0,221500.0,510000.0,107000.0,108500.0,62800.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
**일별 수정주가 데이터를 이용하여 월간 수익률 계산**

In [31]:
return_monthly   = adj_close.resample('ME').last().pct_change(fill_method=None) # 수정주가 데이터로 월별 수익률 계산
return_monthly   = return_monthly.reindex(factors_monthly.index)                        # 팩터 데이터와 인덱스 통일

**`mask`**
- 수정주가 데이터가 없는 구간의 수익률 제거
- 상장폐지된 종목의 구간을 정확히 반영하기 위해 반드시 필요한 작업

In [32]:
# mask 생성
mask = adj_close.notna()
mask = mask.reindex(factors.index)

return_monthly = return_monthly.where(mask)

In [33]:
return_monthly.tail()

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,기아,KB금융,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-30,-0.039792,-0.069219,-0.029895,0.037475,0.271132,-0.033469,0.440860,-0.020585,0.141772,0.234542,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.003049,NaN
2025-05-31,0.012613,0.152113,-0.118644,-0.019011,0.033370,-0.027807,0.008706,-0.011062,0.156319,0.393782,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
2025-06-30,0.064057,0.427873,0.038462,-0.038760,0.045623,0.098219,0.056720,0.083893,0.063279,0.695167,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005097,NaN
2025-07-31,0.193980,-0.063356,0.287879,0.075605,0.174528,0.046683,0.144691,0.055728,0.000000,-0.040936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-31,-0.023810,-0.016453,-0.079739,-0.061856,-0.112450,0.032864,0.060143,0.034213,-0.024346,-0.059451,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
**초과수익률 계산**

In [34]:
# 수익률 분해를 위한 팩터 정리 - 종속변수 설정(Ri - RF)
excess_return = return_monthly.sub(factors_monthly['RF'], axis=0)

In [35]:
excess_return.tail()

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,HD현대중공업,기아,KB금융,두산에너빌리티,...,웨이포트,성융광전투자,완리,골든센츄리,평산차업 KDR,네프로아이티,중국고섬,SBI모기지,SBI핀테크솔루션즈,SNK
Date,,,,,,,,,,,,,,,,,,,,,
2025-04-30,-0.042023,-0.071449,-0.032126,0.035245,0.268901,-0.035699,0.438629,-0.022816,0.139541,0.232311,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.005280,NaN
2025-05-31,0.010479,0.149980,-0.120777,-0.021145,0.031237,-0.029940,0.006573,-0.013195,0.154186,0.391649,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002133,NaN
2025-06-30,0.061948,0.425764,0.036353,-0.040868,0.043514,0.096110,0.054611,0.081784,0.061170,0.693059,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002988,NaN
2025-07-31,0.191912,-0.065424,0.285811,0.073537,0.172460,0.044615,0.142623,0.053660,-0.002068,-0.043004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-31,-0.025894,-0.018538,-0.081823,-0.063940,-0.114534,0.030780,0.058058,0.032129,-0.026431,-0.061535,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
**초과수익률 데이터 저장**

In [36]:
# Output 폴더
excess_return.to_csv("./output/excess_return.csv", index=True, encoding="utf-8-sig")

# 데이터가 필요한 단계의 input 폴더
excess_return.to_csv("../Step2 투자 요인 산출 (Factor construction)/input/excess_return.csv", index=True, encoding="utf-8-sig")